# Homes on Dangerous Streets?
> **Owner:** Mohamed

## Team

| Member | ID |
|---|---|
| Mohamed nazmy | |
| Mostafa Ayman | |
| Mostafa Alaa | |
| Ali | |
| Moustafa Maged | |
| baraa | |

## Contributions

| Member | Contributions |
|---|---|
| Mohamed nazmy | |
| Mostafa Ayman | |
| Mostafa Alaa | |
| Ali | |
| Moustafa Maged | |
| baraa | |

## 1. Introduction
> **Owner:** Mostafa Ayman

### Main question

### Big idea

### Why it matters to New Yorkers

## 2. Datasets and why we chose them
> **Owner:** Mohamed

| | Housing & Development | Transportation |
|---|---|---|
| Dataset | [Affordable Housing Production by Building](https://data.cityofnewyork.us/Housing-Development/Affordable-Housing-Production-by-Building/hg8x-zxpr/about_data) (catalog H2) | [Serious Injury Response Tracking & Analysis Program (SIRTA)](https://data.cityofnewyork.us/Transportation/Serious-Injury-Response-Tracking-Analysis-Program-/xeqp-qz8h/about_data) (catalog T4) |
| Dataset ID | `hg8x-zxpr` | `xeqp-qz8h` |
| Size | 9,551 rows × 41 columns | 13,754 rows × 23 columns |
| One row = | one affordable building | one crash with a severe injury or death |
| Geographic fields | Borough, Postcode (ZIP), Latitude/Longitude | Borough, ZipCode, Latitude/Longitude, street |
| Time fields | Project Start Date, Building Completion Date | Crash Date, Crash Time |

**Why these two.** Our question is whether New York is building its affordable homes on dangerous streets. New affordable buildings add residents who walk, bike and take transit, so the safety of the streets around them matters. The housing dataset tells us where and when affordable buildings were completed and how big they are; SIRTA tells us where and when serious crashes happened. Both are in the course catalog, and both carry the same three geographic fields (borough, ZIP code, coordinates) plus dates, so they can be linked by place and time.

**Size.** Both datasets are small enough to download in full with the loading code from the project description, so no sample was needed.

**Analysis window.** We use 1 Jan 2022 – 30 Jun 2026, the only period both datasets cover. Housing runs from 2014; SIRTA starts in 2022.

**How the datasets will be joined.** A building and a crash never share a record, and the datasets have different row types (buildings vs crashes), so we do not join row to row. Instead we bring both to a common level:

| Level | Used for | Why this level |
|---|---|---|
| ZIP code (and ZIP × year, ZIP × month) | Most questions | Present in both datasets and much finer than borough |
| Borough × year | Q12 | Borough is complete in both datasets, so every record counts |
| Building-level distance (crashes within 100 m / 250 m of a building) | Q8, Q11 | "Doorstep" questions need exact locations, not areas |

**Keys we did not use.** Neighborhood Tabulation Area and census tract use 2010 codes in the housing data but 2020 codes in SIRTA, so they don't match. Council district and community board are mostly empty in SIRTA.

**Known gaps that affect the join** (details in sections 4–7):
- 1,958 housing buildings (20.5%) have no ZIP or coordinates; 1,949 of them are confidential projects. They can be used at borough level only.
- 41.3% of SIRTA crashes have no ZIP code. We plan to fill ZIPs from crash coordinates where possible (section 8).


## 3. Setup
> **Owner:** Mohamed

Imports and constants.

In [1]:
import pandas as pd

# Replace with the dataset IDs found in each dataset's URL
housing_id = 'hg8x-zxpr'
transport_id = 'xeqp-qz8h'

base_url = 'https://data.cityofnewyork.us/api/views/{}/rows.csv?accessType=download'

# Load the chosen datasets
df_housing = pd.read_csv(base_url.format(housing_id), low_memory=False)
df_transport = pd.read_csv(base_url.format(transport_id), low_memory=False)

# Quick preview

df_transport.head()


,Collision ID,Crash Date,Crash Time,Crash Police Precinct,Crash Borough,Total Non-Severe Injuries,Total Severe Injuries,Total Fatalities,Latitude,Longitude,...,Highway Reference Marker,ZipCode,Similar Fatalities & Severe Injuries (3Y Lookback),Similar Fatalities & All Injuries (3Y Lookback),Community Board,Council District,Building Identification Number (BIN),Borough Block Lot (BBL),Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020)
0,4492364.0,01/01/2022,01:28,46.0,Bronx,1.0,1.0,0.0,40.858707,-73.90370,...,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,4491159.0,01/01/2022,04:33,46.0,Bronx,0.0,1.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,4491487.0,01/01/2022,05:55,47.0,Bronx,0.0,1.0,0.0,40.871550,-73.86428,...,NaN,NaN,0.0,0.0,212.0,12.0,2056281.0,2.045980e+09,372.0,BX1201
3,4491223.0,01/01/2022,10:42,43.0,Bronx,0.0,1.0,0.0,40.823673,-73.87233,...,907HX1M11001,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,4491418.0,01/01/2022,20:55,49.0,Bronx,0.0,1.0,0.0,40.863945,-73.86452,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
import sys
sys.path.append("../src")   # lets the notebook import files from src/
import charts

charts.CHARTS["q1"]({}, charts.DEFAULT_FILTERS).show()


## 4. Housing: overview and EDA
> **Owner:** Mohamed

In [2]:
df_housing.head()


,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,2-BR Units,3-BR Units,4-BR Units,5-BR Units,6-BR+ Units,Unknown-BR Units,Counted Rental Units,Counted Homeownership Units,All Counted Units,Total Units
0,68872,177-06 WEXFORD TERRACE,06/30/2026,NaN,991637.0,177-06,WEXFORD TERRACE,Queens,11432.0,4.098350e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,9.0,43
1,72436,ST. NICK'S. 819 GRAND STREET,06/30/2026,NaN,1021347.0,819,GRAND STREET,Brooklyn,11211.0,3.029220e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,187.0,NaN,187.0,187
2,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,21199.0,127,EDGECOMBE AVENUE,Manhattan,10030.0,1.020480e+09,...,20.0,NaN,NaN,NaN,NaN,NaN,25.0,NaN,25.0,25
3,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,28145.0,602,ST NICHOLAS AVENUE,Manhattan,10030.0,1.020480e+09,...,5.0,NaN,NaN,NaN,NaN,NaN,16.0,NaN,16.0,16
4,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,40555.0,129,WEST 133 STREET,Manhattan,10030.0,1.019180e+09,...,6.0,4.0,NaN,NaN,NaN,NaN,15.0,NaN,15.0,15


In [3]:
print(df_housing.shape, df_transport.shape)
print(df_transport["ZipCode"].isna().mean())


(9551, 41) (13754, 23)
0.413116184382725


In [4]:
pd.set_option("display.max_columns", None)
print(df_housing.columns.tolist())


['Project ID', 'Project Name', 'Project Start Date', 'Project Completion Date', 'Building ID', 'Number', 'Street', 'Borough', 'Postcode', 'BBL', 'BIN', 'Community Board', 'Council District', 'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude', 'Longitude', 'Latitude (Internal)', 'Longitude (Internal)', 'Building Completion Date', 'Reporting Construction Type', 'Extended Affordability Only', 'Prevailing Wage Status', 'Extremely Low Income Units', 'Very Low Income Units', 'Low Income Units', 'Moderate Income Units', 'Middle Income Units', 'Other Income Units', 'Studio Units', '1-BR Units', '2-BR Units', '3-BR Units', '4-BR Units', '5-BR Units', '6-BR+ Units', 'Unknown-BR Units', 'Counted Rental Units', 'Counted Homeownership Units', 'All Counted Units', 'Total Units']


In [5]:
missing = pd.DataFrame({
    "dtype": df_housing.dtypes,
    "missing": df_housing.isna().sum(),
    "missing_%": (df_housing.isna().mean() * 100).round(1),
})
missing.sort_values("missing_%", ascending=False)


,dtype,missing,missing_%
6-BR+ Units,float64,9524,99.7
5-BR Units,float64,9478,99.2
4-BR Units,float64,8990,94.1
Unknown-BR Units,float64,8862,92.8
Other Income Units,float64,8191,85.8
Moderate Income Units,float64,7857,82.3
Extremely Low Income Units,float64,7470,78.2
Middle Income Units,float64,6860,71.8
Counted Homeownership Units,float64,6821,71.4
Studio Units,float64,6353,66.5


In [6]:
# Check 1: are the 1,958 rows without a ZIP the same rows without coordinates?
no_postcode = df_housing["Postcode"].isna()
print("No postcode AND no latitude:", (no_postcode & df_housing["Latitude"].isna()).sum())
df_housing.loc[no_postcode, ["Project Name", "Number", "Street", "Borough"]].head(10)


No postcode AND no latitude: 1958


,Project Name,Number,Street,Borough
101,CONFIDENTIAL,----,----,Brooklyn
102,CONFIDENTIAL,----,----,Bronx
103,CONFIDENTIAL,----,----,Bronx
104,CONFIDENTIAL,----,----,Brooklyn
105,CONFIDENTIAL,----,----,Brooklyn
106,CONFIDENTIAL,----,----,Brooklyn
130,CONFIDENTIAL,----,----,Staten Island
131,CONFIDENTIAL,----,----,Bronx
132,CONFIDENTIAL,----,----,Queens
133,CONFIDENTIAL,----,----,Brooklyn


In [7]:
# Check 2: does "empty" mean 0? If the income tiers (empty = 0) add up to
# All Counted Units, empty cells are zeros, not unknown values.
income_cols = ["Extremely Low Income Units", "Very Low Income Units", "Low Income Units",
               "Moderate Income Units", "Middle Income Units", "Other Income Units"]
income_sum = df_housing[income_cols].fillna(0).sum(axis=1)
print("Rows where tiers add up:", (income_sum == df_housing["All Counted Units"]).sum(), "of", len(df_housing))


Rows where tiers add up: 9522 of 9551


In [8]:
# Are all 1,958 location-less rows confidential?
print("CONFIDENTIAL rows:", (df_housing["Project Name"] == "CONFIDENTIAL").sum())

# Categories: how buildings split by borough and construction type
print(df_housing["Borough"].value_counts(), "\n")
print(df_housing["Reporting Construction Type"].value_counts(), "\n")

# Date range of building completions
completion = pd.to_datetime(df_housing["Building Completion Date"], format="%m/%d/%Y")
print("Completions from", completion.min(), "to", completion.max())
print(completion.dt.year.value_counts().sort_index())


CONFIDENTIAL rows: 1949
Borough
Brooklyn         3649
Bronx            2630
Manhattan        1485
Queens           1342
Staten Island     445
Name: count, dtype: int64 

Reporting Construction Type
Preservation        4785
New Construction    4766
Name: count, dtype: int64 

Completions from 2014-01-03 00:00:00 to 2026-08-21 00:00:00
Building Completion Date
2014.0    376
2015.0    290
2016.0    432
2017.0    830
2018.0    664
2019.0    862
2020.0    419
2021.0    581
2022.0    654
2023.0    806
2024.0    705
2025.0    792
2026.0    474
Name: count, dtype: int64


In [9]:
import plotly.express as px

# Summary statistics for the two unit columns
print(df_housing[["All Counted Units", "Total Units"]].describe())

# Box plot on a log scale: the data is strongly skewed, so a normal scale squashes the boxes
px.box(df_housing, x="Borough", y="All Counted Units", log_y=True,
       title="Affordable units per building, by borough (log scale)").show()

# Completions per year (2026 is a partial year, to 21 Aug)
years = completion.dt.year.value_counts().sort_index()
px.bar(x=years.index, y=years.values,
       labels={"x": "Year of building completion", "y": "Buildings completed"},
       title="Affordable buildings completed per year").show()


       All Counted Units  Total Units
count        9522.000000  9551.000000
mean           33.705734    46.653544
std            65.834105    92.482144
min             1.000000     1.000000
25%             2.000000     3.000000
50%             8.000000    12.000000
75%            30.000000    45.000000
max           917.000000  1320.000000


In [10]:
units = df_housing["All Counted Units"]
print("Above IQR fence (72):", (units > 72).sum())

size = df_housing["Total Units"]
print("Small (<50):", (size < 50).sum(),
      "| Medium (50-199):", size.between(50, 199).sum(),
      "| Large (200+):", (size >= 200).sum())


Above IQR fence (72): 1249
Small (<50): 7309 | Medium (50-199): 1698 | Large (200+): 544


In [11]:
# Duplicates: fully identical rows, and repeated building IDs within a project
print("Fully duplicated rows:", df_housing.duplicated().sum())
known_id = df_housing["Building ID"].notna()
print("Repeated Project ID + Building ID:",
      df_housing[known_id].duplicated(subset=["Project ID", "Building ID"]).sum())

# Consistency: affordable units should never exceed total units
print("Affordable > total:", (df_housing["All Counted Units"] > df_housing["Total Units"]).sum())

# Consistency: completion should not come before the project started
start = pd.to_datetime(df_housing["Project Start Date"], format="%m/%d/%Y")
print("Completed before start:", (completion < start).sum())

# Consistency: coordinates should fall inside NYC (roughly lat 40.4–41.0, lon -74.3 to -73.6)
lat, lon = df_housing["Latitude"], df_housing["Longitude"]
print("Outside NYC box:", (~lat.between(40.4, 41.0) & lat.notna()).sum() +
      (~lon.between(-74.3, -73.6) & lon.notna()).sum())


Fully duplicated rows: 0
Repeated Project ID + Building ID: 0
Affordable > total: 0
Completed before start: 20
Outside NYC box: 0


In [12]:
bad_dates = completion < start
check = df_housing.loc[bad_dates, ["Project ID", "Project Name", "Reporting Construction Type",
                                   "Project Start Date", "Building Completion Date"]].copy()
check["days_early"] = (start - completion)[bad_dates].dt.days
check.sort_values("days_early", ascending=False)


,Project ID,Project Name,Reporting Construction Type,Project Start Date,Building Completion Date,days_early
8132,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,01/08/2015,357
8129,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,02/07/2015,327
8127,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,02/26/2015,308
8126,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,04/02/2015,273
8125,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,05/04/2015,241
8135,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,11/30/2015,31
4714,70377,CONFIDENTIAL,New Construction,10/29/2020,10/09/2020,20
1340,76576,CONFIDENTIAL,New Construction,02/27/2025,02/21/2025,6
6423,61837,9 FORT WASHINGTON AVENUE CLUSTER,Preservation,06/25/2018,06/19/2018,6
6422,61837,9 FORT WASHINGTON AVENUE CLUSTER,Preservation,06/25/2018,06/19/2018,6


### 4.x Summary: issues found and what cleaning should do

| # | Issue found | Evidence | Suggested handling | Affects |
|---|---|---|---|---|
| 1 | Confidential buildings have no location | 1,949 rows named `CONFIDENTIAL`; 1,958 rows have no Postcode and no coordinates (same rows) | Keep. Use at borough level only; they drop out of ZIP and distance questions. Check the 9 non-confidential rows without location | Q1–Q11 (ZIP/distance) |
| 2 | Hidden missing values | `Number` and `Street` show 0 missing but use `----` as a placeholder in confidential rows | Replace `----` with missing | Address fields |
| 3 | Empty unit-breakdown cells | Income tiers (empty → 0) add up to `All Counted Units` in every row with a known total | Fill income tiers with 0. Run the same check on bedroom and tenure columns before filling | Q9 |
| 4 | Missing affordable unit count | 29 rows (0.3%) have no `All Counted Units` | Decide: exclude from unit totals, or fill from the tier sum if it is non-zero | Q1, Q7, Q12 |
| 5 | Missing completion date | 1,666 rows (17.4%) have no `Building Completion Date` | Keep; check whether these are unfinished buildings. Exclude from completion-based counts | Q1, Q2, Q7, Q12 |
| 6 | Dates stored as text | All 3 date columns are `str` in `MM/DD/YYYY`; all non-empty completion dates parse with this format | Convert with `pd.to_datetime(..., format="%m/%d/%Y")` | All time questions |
| 7 | ZIP stored as a decimal | `Postcode` is `float64` (`11432.0`) | Convert to a 5-character text ZIP (`"11432"`) | Join with SIRTA |
| 8 | IDs stored as decimals | `Building ID`, `BBL`, `BIN` are `float64` because of missing values | Convert to whole-number or text IDs | Duplicate checks, joins |
| 9 | Very large buildings | 1,249 buildings (13.1%) are above the IQR fence of 72 affordable units; the largest has 917 | Keep: they are real buildings, and Q2/Q6 study large buildings directly. Use domain rules (impossible values) instead of IQR; none found (minimum is 1 unit) | Q2, Q6, all unit totals |
| 10 | Completed before project start | 20 buildings in 5 projects; 14 of them 1–20 days early. Only one is in our window, and it is confidential | Keep and flag; no effect on our questions | Q2, Q6 (none in practice) |
| 11 | Completions after the window | 2026 completions run to 21 Aug 2026; SIRTA ends 30 Jun 2026 | Cut at 30 Jun 2026 | All questions |
| 12 | Two coordinate pairs | `Latitude`/`Longitude` missing in 1,958 rows; the Internal pair in 2,044 | Use `Latitude`/`Longitude`; drop the Internal pair | Q8, Q11 |
| 13 | Columns no question uses | Extended Affordability Only, Prevailing Wage Status, Census Tract, NTA, Community Board, Council District, Internal coordinates | Candidates to drop, to keep the table small | — |

**Checked and fine:** no duplicated rows; no repeated Project ID + Building ID; affordable units never exceed total units; all coordinates fall inside NYC; Borough has exactly 5 clean values; Reporting Construction Type has exactly 2 values (Preservation 4,785, New Construction 4,766).


## 5. SIRTA: overview and EDA
> **Owner:** Ali

## 6. Housing cleaning
> **Owner:** Mostafa Alaa

### 6.1 Missing values

### 6.2 Outliers

### 6.3 Inconsistencies

### 6.4 Standard formats

### 6.5 Duplicates

### 6.6 Before/after check

## 7. SIRTA cleaning
> **Owner:** Ali

### 7.1 Missing values

### 7.2 Outliers

### 7.3 Inconsistencies

### 7.4 Standard formats

### 7.5 Duplicates

### 7.6 Before/after check

## 8. Integration
> **Owner:** Moustafa Maged

### 8.1 Join keys and why

### 8.2 Granularity and aggregation

### 8.3 Spatial join

## 9. Post-integration cleaning and export
> **Owner:** Moustafa Maged

### 9.1 NaNs from joins

### 9.2 Redundant columns

### 9.3 Data types

### 9.4 Export to `data/processed`

## 10. Research questions
> **Owner:** Everyone

### 10.1 Q1
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.2 Q2
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.3 Q3
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.4 Q4
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.5 Q5
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.6 Q6
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.7 Q7
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.8 Q8
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.9 Q9
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.10 Q10
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.11 Q11
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

### 10.12 Q12
> **Owner:**

**Question:**

**Chart:**

**Finding:**

**Limitations:**

## 11. Story summary
> **Owner:** Mostafa Ayman

### Headline findings

### Limitations

### Impact

## 12. Appendix
> **Owner:** Mohamed

### 12.1 Datasets

| Domain | Dataset | Catalog | ID | Link |
|---|---|---|---|---|
| Housing & Development | Affordable Housing Production by Building | H2 | `hg8x-zxpr` | [NYC Open Data](https://data.cityofnewyork.us/Housing-Development/Affordable-Housing-Production-by-Building/hg8x-zxpr/about_data) |
| Transportation | Serious Injury Response Tracking & Analysis Program (SIRTA) | T4 | `xeqp-qz8h` | [NYC Open Data](https://data.cityofnewyork.us/Transportation/Serious-Injury-Response-Tracking-Analysis-Program-/xeqp-qz8h/about_data) |

Both datasets are downloaded in full each time the notebook runs (section 3), so the portal may have newer rows than the numbers in this notebook.

### 12.2 Shared definitions

| Term | Meaning |
|---|---|
| Serious harm | Total Severe Injuries + Total Fatalities, summed over crashes |
| Serious crash | One SIRTA record (SIRTA only records crashes with a severe injury or death) |
| Completed units | All Counted Units (affordable units), dated by Building Completion Date |
| Building size | Total Units (affordable + market-rate) |
| Analysis window | 1 Jan 2022 – 30 Jun 2026; full years = 2022–2025 |

### 12.3 AI prompt log

| # | Date | Member | Tool | Prompt (short) | Used for | What we kept / changed |
|---|---|---|---|---|---|---|
| 1 | | | | | | |
